# 불균형 데이터·임계값·모델 설명

## Bike Sharing 고수요 경보정책 비교

같은 고수요 분류점수에 여러 threshold를 적용해 precision, recall, 경보 수와 오류의 trade-off를 비교합니다.


## 이번 실습에서 완성할 것

- 같은 분류점수에 세 임계값을 적용합니다.
- 경보 수·정밀도·재현율·FP·FN을 함께 읽습니다.
- 순열 중요도를 예측 의존성으로 해석합니다.

**완료 결과:** 임계값 비교표, 운영조건별 후보 기준, 최종 선택에 부족한 비용·용량 정보


## 이번 교시에서 사용할 데이터

### 강의 시연: AI4I 고장점수와 경보정책

AI4I의 10,000개 관측 중 고장은 339건으로 3.39%입니다. 9교시와 같은 사건 정의·입력변수·Logistic Regression 명세를 사용하되, 이 교시에서는 development 안에 fit과 validation을 추가해 정책 탐색용 자료를 분리합니다. 모형 명세는 바꾸지 않고, 재현한 validation 분류점수 벡터에 여러 threshold만 적용합니다. 균형 가중치를 사용한 출력은 실제 위험확률이 아니라 경보 순위를 위한 점수로 다룹니다.

- **positive class:** `Machine failure = 1`
- **정책 비교값:** threshold별 alerts, precision, recall, FP, FN
- **설명 입력:** 온도, 회전속도, torque, tool wear 등 결과 label이 아닌 feature
- **주의:** prevalence가 달라지면 precision과 실제 경보부담도 달라질 수 있음

### 적용 실습: Bike 고수요 경보

Bike 시간별 자료의 고수요 분류점수에 여러 threshold를 적용합니다. 운영 인력이 처리할 수 있는 경보 수와 고수요를 놓치는 비용을 가정해 후보 threshold를 좁힙니다. 수치가 없는 경우에는 최종값을 고르지 않고 필요한 정보를 남깁니다. AI4I와 Bike의 event 정의는 다르지만, 분류점수와 경보 threshold를 분리하는 절차는 같습니다.


## 실습에서 고정할 연구조건

| 그대로 유지할 방법 | 이번 실습에서 판단할 내용 |
|---|---|
| 9교시와 같은 사건·입력·모형 명세 | validation의 세 threshold와 오류·경보량 |
| fit에서 학습하고 validation에서 비교 | 놓친 사건과 경보부담 중 어떤 조건을 우선할지 |
| 최종 test와 test 점수는 봉인 | 최종 선택에 부족한 실제 비용·처리용량 |
| 순열 중요도도 validation에서 계산 | 예측 의존성과 인과효과를 구분 |

비용비율과 하루 처리용량이 없으면 최적 threshold를 선언하지 않습니다.


## 실제 실행 순서와 산출물

1. development를 fit과 validation으로 나누고 최종 test가 봉인되었는지 확인합니다.
2. fit에서 모형을 학습하고 validation 사건비율과 AP를 확인합니다.
3. validation의 0.20·0.50·0.80에서 경보량·precision·recall·FP·FN을 비교합니다.
4. validation의 PR 곡선과 순열 중요도를 읽고, 최종 선택에 부족한 정보를 기록합니다.

**산출물:** threshold 비교표, PR 곡선, 순열 중요도, 후보 기준과 정보 부족 기록


## 실행 환경


In [ ]:
from pathlib import Path
import platform
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 130)
warnings.filterwarnings("ignore", category=FutureWarning)

def find_course_data(relative_path: str) -> Path:
    """0교시에서 준비한 data 폴더의 파일을 찾습니다."""
    relative = Path(relative_path)
    checked = []
    for start in [Path.cwd(), *Path.cwd().parents, Path("/content")]:
        candidate = (start / "data" / relative).resolve()
        if candidate in checked:
            continue
        checked.append(candidate)
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"데이터 파일을 찾지 못했습니다: {relative_path}. "
        "과정 폴더의 00_setup.ipynb를 먼저 실행해 주세요."
    )
print("Python:", platform.python_version())
print("pandas:", pd.__version__)


## 최종 test를 봉인하는 이유

threshold와 순열 중요도는 **validation**에서 비교합니다. 수업에는 실제 오류비용과 경보 처리용량이 없으므로 후보의 trade-off와 부족한 정보만 기록하고 최종 threshold를 확정하지 않습니다. 따라서 기본 경로에서는 test 점수를 계산하지 않습니다.

실제 연구에서 비용·용량을 사전에 정해 정책 threshold를 고른 경우에만 다음 순서를 따릅니다.

1. validation에서 정책 threshold를 한 번 고정합니다.
2. fit과 validation을 합친 development 자료로 같은 모형을 다시 적합합니다.
3. 봉인한 test를 한 번만 평가합니다.
4. test 결과를 보고 threshold나 모형을 다시 바꾸지 않습니다.


In [ ]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, confusion_matrix, precision_recall_curve, precision_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

data = pd.read_csv(find_course_data("bike/hour.csv"), parse_dates=["dteday"])
data["timestamp"] = data["dteday"] + pd.to_timedelta(data["hr"], unit="h")
data = data.sort_values("timestamp").reset_index(drop=True)

# 마지막 20%는 최종 test로 봉인합니다. 앞 80% 안에서 fit과 validation을 시간순으로 나눕니다.
development_cut = int(len(data) * 0.80)
development = data.iloc[:development_cut].copy()
test_sealed = data.iloc[development_cut:].copy()
fit_cut = int(len(development) * 0.80)
fit = development.iloc[:fit_cut].copy()
validation = development.iloc[fit_cut:].copy()

HIGH_DEMAND_QUANTILE = 0.85  # 9교시와 같은 train-derived 사건 정의
event_cutoff = float(development["cnt"].quantile(HIGH_DEMAND_QUANTILE))
target_name = "high_demand"
for frame in [fit, validation, test_sealed]:
    frame[target_name] = frame["cnt"].gt(event_cutoff).astype(int)

numeric_features = ["temp", "atemp", "hum", "windspeed"]
categorical_features = ["season", "mnth", "hr", "holiday", "weekday", "workingday", "weathersit"]
feature_columns = numeric_features + categorical_features
analysis_name = "Bike high-demand threshold policy"
X_fit, y_fit = fit[feature_columns], fit[target_name]
X_validation, y_validation = validation[feature_columns], validation[target_name]
X_test_sealed, y_test_sealed = test_sealed[feature_columns], test_sealed[target_name]

preprocess = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
])
model = Pipeline([
    ("prep", clone(preprocess)),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])
model.fit(X_fit, y_fit)
classification_score = model.predict_proba(X_validation)[:, 1]

TEST_SCORED = False
POLICY_THRESHOLD = None
MAX_ALERTS_PER_DAY = None
FN_TO_FP_COST_RATIO = None

print("event cutoff from development:", event_cutoff)
print("fit period:", fit["timestamp"].min(), "→", fit["timestamp"].max(), "| rows:", len(fit))
print("validation period:", validation["timestamp"].min(), "→", validation["timestamp"].max(), "| rows:", len(validation))
print("sealed test starts:", test_sealed["timestamp"].min(), "| rows:", len(test_sealed))
print("fit/validation prevalence:", y_fit.mean(), y_validation.mean())
print("validation average precision:", average_precision_score(y_validation, classification_score))
print("최종 test의 label·점수·성능은 아직 열지 않았습니다.")


## 2. 세 임계값의 경보량과 오류 비교

threshold를 높이면 경보 수와 recall은 같거나 감소합니다. precision은 반드시 계속 증가하지 않으므로 세 후보의 실제 결과표에서 방향을 확인합니다.


In [ ]:
rows = []
for threshold in [0.20, 0.50, 0.80]:
    pred = (classification_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_validation, pred, labels=[0, 1]).ravel()
    rows.append({
        "threshold": threshold,
        "alerts": int(pred.sum()),
        "precision": precision_score(y_validation, pred, zero_division=0),
        "recall": recall_score(y_validation, pred, zero_division=0),
        "FP": fp, "FN": fn, "TP": tp, "TN": tn,
    })
result_table = pd.DataFrame(rows)
result_table.insert(0, "selection_split", "validation")
validation_days = max((validation['timestamp'].max() - validation['timestamp'].min()).total_seconds() / 86400, 1)
result_table['alerts_per_day'] = result_table['alerts'] / validation_days
display(result_table)
MAX_ALERTS_PER_DAY = None  # 실제 운영값을 알 때 숫자로 바꿉니다.
FN_TO_FP_COST_RATIO = None  # 실제 비용비율을 알 때 숫자로 바꿉니다.
print('운영 용량·비용 수치가 없으므로 지금은 후보 비교만 하고 최종 threshold는 선택하지 않습니다.')
toy_tp, toy_fp, toy_fn = 4, 6, 1
print(f"\nprecision = {toy_tp}/({toy_tp}+{toy_fp}) = {toy_tp/(toy_tp+toy_fp):.2f}  # 경보가 분모")
print(f"recall    = {toy_tp}/({toy_tp}+{toy_fn}) = {toy_tp/(toy_tp+toy_fn):.2f}  # 실제 사건이 분모")
print("\n[threshold별 경보정책을 문장으로 읽기]")
for row in result_table.itertuples(index=False):
    print(f"- threshold {row.threshold:.2f}: 경보 {int(row.alerts)}건, precision {row.precision:.1%}, recall {row.recall:.1%}, FP {int(row.FP)}, FN {int(row.FN)}")


## 3. PR 곡선·AP·순열 중요도 읽기

PR 곡선은 threshold가 바뀔 때 precision과 recall이 함께 어떻게 달라지는지 보여 줍니다. **Average Precision(AP)**은 여러 threshold에서의 PR 성능을 하나로 요약한 값이며, 특정 threshold 하나의 precision과는 다릅니다. 무작위 순위의 AP 기준은 대체로 평가자료의 사건비율입니다.

순열 중요도는 입력 열 하나를 섞었을 때 AP가 얼마나 낮아지는지 측정합니다. 현재 모형의 예측 의존성을 보여 주지만 그 변수가 사건의 원인이라는 뜻은 아닙니다.


In [ ]:
import matplotlib.pyplot as plt

precision_curve, recall_curve, _ = precision_recall_curve(y_validation, classification_score)
fig, ax = plt.subplots(figsize=(6.8, 4.5))
ax.plot(recall_curve, precision_curve, linewidth=2)
ax.axhline(y_validation.mean(), color="gray", linestyle="--", label="prevalence baseline")
ax.set(title="Precision–Recall curve", xlabel="Recall", ylabel="Precision", xlim=(0, 1), ylim=(0, 1))
ax.legend()
plt.tight_layout()
plt.show()

importance = permutation_importance(model, X_validation, y_validation, scoring="average_precision", n_repeats=5, random_state=42, n_jobs=1)
importance_table = pd.DataFrame({"feature": feature_columns, "importance_mean": importance.importances_mean, "importance_sd": importance.importances_std}).sort_values("importance_mean", ascending=False)
display(importance_table.head(10))
point_indices = np.unique(np.linspace(0, len(precision_curve) - 1, 6, dtype=int))
pr_point_view = pd.DataFrame({
    "recall": recall_curve[point_indices],
    "precision": precision_curve[point_indices],
})
print("\n[PR curve의 대표 지점]")
display(pr_point_view.round(3))
print(f"무작위 경보의 precision 기준은 validation 사건비율 {y_validation.mean():.1%} 부근입니다.")

top_importance = importance_table.head(5).copy()
top_importance["섞은 뒤 예상 점수"] = average_precision_score(y_validation, classification_score) - top_importance["importance_mean"]
print("[열을 섞었을 때 average precision 감소가 큰 순서]")
display(top_importance[["feature", "importance_mean", "섞은 뒤 예상 점수"]].round(3))
print("이 값은 모델의 예측 의존성이며 인과효과가 아닙니다.")


## 4. 경보 기준 비교표의 검증

threshold가 증가할수록 alerts가 감소하는지, 분류점수와 importance가 유한한지 확인합니다.


In [ ]:
checks = pd.Series({
    "selection uses validation": set(result_table["selection_split"]) == {"validation"},
    "sealed test untouched": TEST_SCORED is False and POLICY_THRESHOLD is None,
    "classification score range": ((classification_score >= 0) & (classification_score <= 1)).all(),
    "three policies": len(result_table) == 3,
    "daily alert burden finite": np.isfinite(result_table["alerts_per_day"]).all(),
    "alerts decrease": result_table["alerts"].is_monotonic_decreasing,
    "finite policy metrics": np.isfinite(result_table[["precision", "recall"]]).all().all(),
    "all raw features explained": len(importance_table) == len(feature_columns),
}, name="passed")
display(checks.to_frame())
assert checks.all()
print("경보 기준 비교 검증: PASS")


## 연구 기록

1. **fit·validation·봉인 test의 역할과 행 수:**
2. **validation의 사건비율과 AP:**
3. **validation에서 비교한 세 threshold의 경보 수·하루 평균 경보 수·precision·recall·FP·FN:**
4. **놓친 사건을 우선할 때 후보로 남길 threshold와 근거:**
5. **경보 처리용량을 우선할 때 후보로 남길 threshold와 근거:**
6. **최종 선택 전에 필요한 실제 비용비율·하루 처리용량:**
7. **순열 중요도로 말할 수 있는 것과 말할 수 없는 것:**

비용과 처리용량의 수치가 없다면 하나의 최적 threshold를 선언하지 않습니다.

### 선택 검토: ChatGPT에 물어볼 때

```text
fit·validation·봉인 test의 역할, validation 사건비율과 threshold별 경보 수·하루 평균 경보 수·precision·recall·FP·FN을 아래에 제공합니다. 모든 비교가 validation에서 이루어졌고 test가 열리지 않았는지 먼저 확인해 주세요. 두 운영상황에서 후보를 좁히되, 비용비율이나 하루 처리용량이 없으면 최종 threshold를 고르지 말고 필요한 정보를 적어 주세요.
```


## 완료 확인

마지막 출력이 `경보 기준 비교 검증: PASS`인지 확인해 주세요. 이 검사는 필요한 계산이 실행됐는지 확인할 뿐, 연구 주장의 타당성을 자동 판정하지 않습니다.


## 결과를 정리할 질문

- 놓친 사건 비용과 처리 가능한 경보 수를 어떻게 가정했습니까?
- 세 임계값에서 경보 수·precision·recall·FP·FN은 어떻게 달라졌습니까?
- 이번 비교는 조건부 정책 탐색이며, 최종 test는 아직 봉인되어 있으며, 비용·용량으로 정책을 먼저 정한 뒤 한 번만 평가해야 한다는 한계를 기록했습니까?
